In [4]:
import pandas as pd
pd.set_option('display.max_column', 20)

df = pd.read_excel('https://storage.googleapis.com/dqlab-dataset/cth_churn_analysis_train.xlsx')
df.head()

,ID_Customer,Jenis_kelamin,umur,membership_program,using_reward,pembayaran,Subscribe_brochure,harga_per_bulan,jumlah_harga_langganan,churn
0,1005-CTMP,Perempuan,41,36,No,Cash,No,10000,360000,Yes
1,1007-STSJ,Laki-laki,27,36,Yes,Bank Transfer,Email,10000,360000,Yes
2,1012-NCGH,Laki-laki,45,36,No,Cash,No,10000,360000,No
3,1014-WWBN,Perempuan,33,12,No,Bank Transfer,Yes,10000,120000,No
4,1024-HYUA,Perempuan,38,12,No,Cash,No,10000,120000,No


In [5]:
df.drop('ID_Customer', axis=1, inplace=True)
df.drop('harga_per_bulan', axis=1, inplace=True)
df.drop('jumlah_harga_langganan', axis=1, inplace=True)

In [6]:
y = df.pop('churn').to_list()
y = [1 if label == 'Yes' else 0 for label in y]

In [7]:
from sklearn.preprocessing import LabelEncoder

labelers = {}

column_categorical_non_binary = []
for col in df.select_dtypes(include=['object']):
	#saat jumlah nilai unik dari suatu kolom sama dengan dua
	#atau dengan kata lain kolom bersifat biner
	if len(df[col].unique()) == 2:
		#buat objek LabelEncoder baru untuk kolom dan tampung dalam
		#dictionary labelers
		labelers[col] = LabelEncoder()
		#meminta objek LabelEncoder untuk mempelajari dan
		#mentransformasikan kolom
		df[col] = labelers[col].fit_transform(df[col])
	#untuk kolom bersifat non-biner
	else:
		#tambahkan nama kolom ke dalam array yang telah disiapkan
		column_categorical_non_binary.append(col)		

df.head()

,Jenis_kelamin,umur,membership_program,using_reward,pembayaran,Subscribe_brochure
0,1,41,36,0,Cash,No
1,0,27,36,1,Bank Transfer,Email
2,0,45,36,0,Cash,No
3,1,33,12,0,Bank Transfer,Yes
4,1,38,12,0,Cash,No


In [8]:
df = pd.get_dummies(df, columns=column_categorical_non_binary)
df.head()

,Jenis_kelamin,umur,membership_program,using_reward,pembayaran_Bank Transfer,pembayaran_Cash,pembayaran_Credit Card,Subscribe_brochure_Email,Subscribe_brochure_No,Subscribe_brochure_Yes
0,1,41,36,0,False,True,False,False,True,False
1,0,27,36,1,True,False,False,True,False,False
2,0,45,36,0,False,True,False,False,True,False
3,1,33,12,0,True,False,False,False,False,True
4,1,38,12,0,False,True,False,False,True,False


In [11]:
X = df.to_numpy()
 
print("Dimensi dari variabel X:", X.shape)

Dimensi dari variabel X: (499, 10)


In [14]:
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, random_state=23)

print("Dimensi data X mula-mula:", X.shape)
print("Dimensi data y mula-mula:", len(y),"\n")
 
print("Dimensi data X train:", X_train.shape)
print("Dimensi data y train:", len(y_train),"\n")
 
print("Dimensi data X test:", X_test.shape)
print("Dimensi data y test:", len(y_test),"\n") 

Dimensi data X mula-mula: (499, 10)
Dimensi data y mula-mula: 499 

Dimensi data X train: (449, 10)
Dimensi data y train: 449 

Dimensi data X test: (50, 10)
Dimensi data y test: 50 



In [15]:
from sklearn.tree import DecisionTreeClassifier
 
model = DecisionTreeClassifier(random_state=57)
 
print("Memulai melatih 'model'.")
model.fit(X_train, y_train)
print("Selesai melatih 'model'.")

Memulai melatih 'model'.
Selesai melatih 'model'.


In [17]:
from sklearn.metrics import accuracy_score

#melakukan prediksi terhadap data latih dan menampung
#label hasil prediksi dalam variabel y_pred
y_pred = model.predict(X_train)
score = accuracy_score(y_train, y_pred)
print("Skor akurasi untuk data latih:", score)

#melakukan prediksi terhadap data testing dan menampung
#label hasil prediksi dalam variabel y_pred
y_pred = model.predict(X_test)
score = accuracy_score(y_test,y_pred)
print("Skor akurasi untuk data testing:", score)

Skor akurasi untuk data latih: 0.9710467706013363
Skor akurasi untuk data testing: 0.5


In [18]:
#mendefinisikan nilai dari parameter 'min_samples_split' yang akan dicobakan
min_samples_split_search = [2,4,8,16,32,64]

#mendefinisikan nilai dari parameter 'max_depth' yang akan dicobakan
max_depth_search = [4,8,16,32,64,128]

#mendefinisikan variabel untuk menyimpan skor terbaik dari setiap model dengan parameter yang berbeda
max_score = 0

#mendefinisikan variabel untuk menyimpan model terbaik.
best_model = None

#mencoba membuat model DecisionTree berdasarkan nilai kombinasi
#dari parameter 'min_samples_split' dan 'max_depth'
for ms in min_samples_split_search:
	for md in max_depth_search:
		# menginisialisasi model berdasarkan salah satu kombinasi nilai yang ada
		model = DecisionTreeClassifier(min_samples_split=ms, max_depth=md, random_state=57)
		
		#melatih model yang telah diinisialisasi dengan data 
		#X_train dan label y_train
		model.fit(X_train, y_train)
		
		#melakukan prediksi terhadap data X_test
		y_pred = model.predict(X_test)
		
		#menghitung skor berdasarkan nilai aktual (y_test) dan (y_pred)
		score = accuracy_score(y_test, y_pred)
		
		#jika score yang dihasilkan oleh model lebih besar dari skor
		#terbesar yang dicatat (max_score), maka
		if max_score < score:
			#simpan model dalam variabel best_model
			best_model = model
			
			#update max_score menjadi score milik model
			max_score = score
			
print("Skor testing terbaik: ", max_score)
print("Parameter model: max_depth=",  best_model.get_params()['max_depth'],", min_samples_split=",
	  best_model.get_params()['min_samples_split'])

Skor testing terbaik:  0.58
Parameter model: max_depth= 8 , min_samples_split= 32


In [19]:
#metode RandomForestClassifier dapat diakses pada library
#scikit-learn, tepatnya pada modul ensemble.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
 
#menginisialisasi model dengan default parameter
model = RandomForestClassifier(random_state=57)
 
#melatih model dengan menggunakan data training
model.fit(X_train, y_train)
 
#meminta model yang telah dilatih melakukan prediksi
#terhadap data latih dan menghitung akurasi prediksi
y_pred = model.predict(X_train)
score = accuracy_score(y_train, y_pred)
 
print("Akurasi untuk data training: ", score)

#meminta model yang telah dilatih melakukan prediksi
#terhadap data testing dan menghitung akurasi prediksi
y_pred = model.predict(X_test)
score = accuracy_score(y_test,y_pred)
 
print("Akurasi untuk data testing: ", score)

Akurasi untuk data training:  0.9710467706013363
Akurasi untuk data testing:  0.48


In [20]:
#parameter untuk mengatur setiap Decision Tree yang akan dibentuk pada model Random Forest
min_samples_split_search = [8, 12, 16, 20,24]
max_depth_search = [4,5,6,7,8]
 
#parameter untuk mengatur jumlah model Decision Tree yang akan terbentuk pada model Random Forest
n_estimators_search = [10, 25, 50, 75, 100]
 
max_score = 0
best_model = None
for ms in min_samples_split_search:
	for md in max_depth_search:
		for ne in n_estimators_search:
			model = RandomForestClassifier(n_estimators = ne, min_samples_split=ms, max_depth=md, random_state=57)
			model.fit(X_train, y_train)
			y_pred = model.predict(X_test)
			score = accuracy_score(y_test,y_pred)
			if max_score < score:
				best_model = model
				max_score = score

print("Skor testing terbaik: ", max_score)
print("Parameter model: max_depth=", 
      best_model.get_params()['max_depth'], 
      ", min_samples_split=",
      best_model.get_params()['min_samples_split'],
      ", n_estimators=",
      best_model.get_params()['n_estimators']
      )

Skor testing terbaik:  0.56
Parameter model: max_depth= 6 , min_samples_split= 16 , n_estimators= 75
